# Movie Dataset Collection - Part 1

**Team members:** Idan Buzaglo, ID 211715131; Mika Melamed, ID 207488453.  
**Git repository:** https://github.com/buzagloidan/movie-data-assignment

Our assigned team letter is `M`, so this notebook collects IMDb movies whose `primaryTitle` starts with `M`. IMDb's free non-commercial TSV files provide the core movie fields and actor ids. English Wikipedia is used to enrich budget, box office, language, country and plot where a usable page is available.


## Collection Code

The full collection script is included below for review. It downloads IMDb TSV files when they are missing, otherwise it reuses the cached files in `imdb_data/`. Running the next code cell writes the same code to `collect_movies.py`, so the notebook and script stay consistent.


In [ ]:
from pathlib import Path

collect_movies_source = '#!/usr/bin/env python3\n"""\nCollect the movie dataset for assignment part 1.\n\nThe team letter is M, so the base population is IMDb movies whose primary\ntitle starts with M. IMDb\'s public TSV files provide the required title,\nrating, vote and actor-id fields. English Wikipedia pages are used for budget,\nbox office, language, country and plot enrichment.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport json\nimport math\nimport os\nimport re\nimport ssl\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nfrom pathlib import Path\n\n\nBASE_DIR = Path(__file__).resolve().parent\nCACHE_DIR = Path(os.environ.get("CACHE_DIR", BASE_DIR / "cache"))\nRAW_DIR = Path(os.environ.get("RAW_DIR", BASE_DIR / "imdb_data"))\nDATASET_PATH = Path(os.environ.get("DATASET_PATH", BASE_DIR / "dataset.csv"))\nREPORT_STATS_PATH = Path(os.environ.get("REPORT_STATS_PATH", BASE_DIR / "report_stats.json"))\n\nTEAM_LETTER = os.environ.get("TEAM_LETTER", "M").strip()[:1].upper() or "M"\nTARGET_MOVIE_COUNT = int(os.environ.get("TARGET_MOVIE_COUNT", "10000"))\nCANDIDATE_POOL_SIZE = int(os.environ.get("CANDIDATE_POOL_SIZE", str(TARGET_MOVIE_COUNT * 3)))\nREQUEST_SLEEP_SECONDS = float(os.environ.get("REQUEST_SLEEP_SECONDS", "0.25"))\nMAX_RATE_LIMIT_RETRIES = int(os.environ.get("MAX_RATE_LIMIT_RETRIES", "2"))\nREQUIRE_WIKIPEDIA_PAGE = os.environ.get("REQUIRE_WIKIPEDIA_PAGE", "0").strip().lower() not in {"0", "false", "no"}\nWIKIPEDIA_ENRICH_LIMIT = int(os.environ.get("WIKIPEDIA_ENRICH_LIMIT", str(TARGET_MOVIE_COUNT)))\n\nUSER_AGENT = "movie-data-assignment/1.0 (student project)"\nRATE_LIMITED_HOSTS: set[str] = set()\n\nIMDB_FILES = {\n    "basics": "title.basics.tsv.gz",\n    "ratings": "title.ratings.tsv.gz",\n    "principals": "title.principals.tsv.gz",\n}\n\nFIELDNAMES = [\n    "tconst",\n    "primaryTitle",\n    "startYear",\n    "genres",\n    "lead_actors_ids",\n    "runtimeMinutes",\n    "averageRating",\n    "Language",\n    "Country",\n    "numVotes",\n    "budget",\n    "BoxOffice",\n    "plot",\n]\n\n\ndef read_json(path: Path) -> dict:\n    if path.exists():\n        try:\n            return json.loads(path.read_text(encoding="utf-8"))\n        except json.JSONDecodeError:\n            return {}\n    return {}\n\n\ndef write_json(path: Path, data: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")\n\n\ndef open_json_request(req: urllib.request.Request, context: ssl.SSLContext | None = None) -> dict:\n    if context is None:\n        with urllib.request.urlopen(req, timeout=30) as response:\n            return json.loads(response.read().decode("utf-8"))\n    with urllib.request.urlopen(req, timeout=30, context=context) as response:\n        return json.loads(response.read().decode("utf-8"))\n\n\ndef get_json(url: str, cache_name: str, sleep: float = REQUEST_SLEEP_SECONDS) -> dict:\n    cache_path = CACHE_DIR / cache_name\n    cached = read_json(cache_path)\n    if cached:\n        return cached\n\n    host = urllib.parse.urlparse(url).netloc\n    if host in RATE_LIMITED_HOSTS:\n        return {}\n\n    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})\n    for attempt in range(1, MAX_RATE_LIMIT_RETRIES + 1):\n        try:\n            payload = open_json_request(req)\n            break\n        except urllib.error.HTTPError as exc:\n            if exc.code != 429:\n                raise\n            wait_seconds = int(exc.headers.get("Retry-After") or 30 * attempt)\n            print(f"Rate limited by {host}; waiting {wait_seconds} seconds before retry", flush=True)\n            time.sleep(wait_seconds)\n        except urllib.error.URLError as exc:\n            if "CERTIFICATE_VERIFY_FAILED" not in str(exc):\n                raise\n            context = ssl._create_unverified_context()\n            try:\n                payload = open_json_request(req, context=context)\n                break\n            except urllib.error.HTTPError as http_exc:\n                if http_exc.code != 429:\n                    raise\n                wait_seconds = int(http_exc.headers.get("Retry-After") or 30 * attempt)\n                print(f"Rate limited by {host}; waiting {wait_seconds} seconds before retry", flush=True)\n                time.sleep(wait_seconds)\n    else:\n        return {}\n\n    write_json(cache_path, payload)\n    time.sleep(sleep)\n    return payload\n\n\ndef download_file(url: str, path: Path) -> None:\n    if path.exists() and path.stat().st_size > 0:\n        return\n\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp_path = path.with_suffix(path.suffix + ".tmp")\n    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})\n    print(f"Downloading {url}", flush=True)\n    with urllib.request.urlopen(req, timeout=120) as response, tmp_path.open("wb") as out:\n        while True:\n            chunk = response.read(1024 * 1024)\n            if not chunk:\n                break\n            out.write(chunk)\n    tmp_path.replace(path)\n\n\ndef ensure_imdb_files() -> dict[str, Path]:\n    paths = {}\n    for key, filename in IMDB_FILES.items():\n        path = RAW_DIR / filename\n        download_file(f"https://datasets.imdbws.com/{filename}", path)\n        paths[key] = path\n    return paths\n\n\ndef iter_tsv_gz(path: Path):\n    with gzip.open(path, "rt", encoding="utf-8", newline="") as f:\n        yield from csv.DictReader(f, delimiter="\\t")\n\n\ndef clean_int(value: str | None) -> int | None:\n    if not value or value == r"\\N" or value == "N/A":\n        return None\n    match = re.search(r"\\d[\\d,]*", value)\n    if not match:\n        return None\n    return int(match.group(0).replace(",", ""))\n\n\ndef clean_float(value: str | None) -> float | None:\n    if not value or value == r"\\N" or value == "N/A":\n        return None\n    try:\n        return float(value.replace(",", ""))\n    except ValueError:\n        return None\n\n\ndef normalize_list_field(value: str | None) -> str:\n    if not value or value == r"\\N" or value == "N/A":\n        return "[]"\n    parts = [part.strip() for part in value.split(",") if part.strip()]\n    return json.dumps(parts, ensure_ascii=False)\n\n\ndef first_value(value: str | None) -> str:\n    if not value or value == "N/A":\n        return ""\n    return value.split(",")[0].strip()\n\n\ndef parse_money_to_millions(text: str | None) -> float | None:\n    if not text:\n        return None\n\n    text = clean_wiki_text(text)\n    if "N/A" in text:\n        return None\n\n    values = []\n    for raw_amount, raw_unit in re.findall(\n        r"(?:US\\$|\\$|USD\\s*)\\s*([0-9][0-9,]*(?:\\.[0-9]+)?)\\s*(billion|million|m)?",\n        text,\n        flags=re.I,\n    ):\n        amount = float(raw_amount.replace(",", ""))\n        unit = raw_unit.lower()\n        if unit.startswith("b"):\n            amount *= 1000\n        elif unit in ("", "m") and ("," in raw_amount or amount > 10000):\n            amount /= 1_000_000\n        values.append(amount)\n\n    if not values:\n        return None\n    return round(sum(values) / len(values), 2)\n\n\ndef safe_name(value: str) -> str:\n    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value)[:120]\n\n\ndef build_imdb_candidates(paths: dict[str, Path]) -> list[dict]:\n    print(f"Reading IMDb basics for movies starting with {TEAM_LETTER}", flush=True)\n    candidates: dict[str, dict] = {}\n    for row in iter_tsv_gz(paths["basics"]):\n        title = row.get("primaryTitle", "").strip()\n        if row.get("titleType") != "movie":\n            continue\n        if not title.upper().startswith(TEAM_LETTER):\n            continue\n\n        year = clean_int(row.get("startYear"))\n        runtime = clean_int(row.get("runtimeMinutes"))\n        if year is None or year > 2024:\n            continue\n        if runtime is None or not 60 <= runtime <= 300:\n            continue\n\n        candidates[row["tconst"]] = {\n            "tconst": row["tconst"],\n            "primaryTitle": title,\n            "startYear": year,\n            "genres": normalize_list_field(row.get("genres")),\n            "runtimeMinutes": runtime,\n        }\n\n    print(f"Found {len(candidates)} eligible {TEAM_LETTER}-title movies before ratings", flush=True)\n    print("Joining IMDb ratings", flush=True)\n    for row in iter_tsv_gz(paths["ratings"]):\n        candidate = candidates.get(row.get("tconst"))\n        if not candidate:\n            continue\n        candidate["averageRating"] = clean_float(row.get("averageRating"))\n        candidate["numVotes"] = clean_int(row.get("numVotes"))\n\n    rated = [\n        item for item in candidates.values()\n        if item.get("averageRating") is not None and item.get("numVotes") is not None\n    ]\n    rated.sort(key=lambda item: (-item["numVotes"], item["primaryTitle"].casefold(), item["tconst"]))\n    pool = rated[:CANDIDATE_POOL_SIZE]\n    print(f"Selected {len(pool)} rated candidates for Wikipedia enrichment", flush=True)\n    return pool\n\n\ndef attach_actor_ids(paths: dict[str, Path], candidates: list[dict]) -> None:\n    selected_ids = {row["tconst"] for row in candidates}\n    actor_ids = {tconst: [] for tconst in selected_ids}\n    remaining = set(selected_ids)\n    current_selected: str | None = None\n\n    print("Reading IMDb principals for lead actor ids", flush=True)\n    for row in iter_tsv_gz(paths["principals"]):\n        tconst = row.get("tconst")\n\n        if current_selected and tconst != current_selected:\n            remaining.discard(current_selected)\n            current_selected = None\n            if not remaining:\n                break\n\n        if tconst not in remaining:\n            continue\n\n        current_selected = tconst\n        if row.get("category") != "actor":\n            continue\n        values = actor_ids[tconst]\n        nconst = row.get("nconst", "")\n        if nconst and nconst not in values and len(values) < 5:\n            values.append(nconst)\n\n    for row in candidates:\n        row["lead_actors_ids"] = json.dumps(actor_ids.get(row["tconst"], []), ensure_ascii=False)\n\n\ndef wikipedia_search(title: str, year: int | None) -> str | None:\n    query = f\'{title} {year} film\' if year else f"{title} film"\n    params = urllib.parse.urlencode({\n        "action": "query",\n        "list": "search",\n        "srsearch": query,\n        "format": "json",\n        "srlimit": 3,\n        "utf8": 1,\n    })\n    data = get_json(f"https://en.wikipedia.org/w/api.php?{params}", f"wiki_search_{safe_name(query)}.json")\n    for result in data.get("query", {}).get("search", []):\n        page_title = result.get("title", "")\n        if page_title:\n            return page_title\n    return None\n\n\ndef wikipedia_page_fields(title: str) -> dict:\n    params = urllib.parse.urlencode({\n        "action": "parse",\n        "page": title,\n        "prop": "wikitext",\n        "format": "json",\n        "redirects": 1,\n    })\n    data = get_json(f"https://en.wikipedia.org/w/api.php?{params}", f"wiki_parse_{safe_name(title)}.json")\n    if "error" in data:\n        return {}\n    wikitext = data.get("parse", {}).get("wikitext", {}).get("*", "")\n    if not wikitext:\n        return {}\n    fields = extract_infobox_fields(wikitext)\n    fields["plot"] = extract_plot(wikitext)\n    return fields\n\n\ndef extract_infobox_fields(wikitext: str) -> dict:\n    wanted = {\n        "budget": "budget",\n        "gross": "BoxOffice",\n        "language": "Language",\n        "country": "Country",\n    }\n    result = {}\n    active_key = None\n    active_value: list[str] = []\n\n    def finish_active() -> None:\n        if not active_key:\n            return\n        cleaned = clean_wiki_text(" ".join(active_value))\n        if active_key in ("Language", "Country"):\n            cleaned = first_value(cleaned)\n        result[active_key] = cleaned\n\n    for line in wikitext.splitlines():\n        line = line.strip()\n        if line.startswith("|") and "=" in line:\n            finish_active()\n            key, value = line[1:].split("=", 1)\n            active_key = wanted.get(key.strip().lower())\n            active_value = [value]\n            continue\n        if active_key and not line.startswith("}}"):\n            active_value.append(line)\n\n    finish_active()\n    return result\n\n\ndef extract_plot(wikitext: str) -> str:\n    patterns = [\n        r"^==\\s*Plot\\s*==\\s*$",\n        r"^==\\s*Synopsis\\s*==\\s*$",\n        r"^==\\s*Premise\\s*==\\s*$",\n    ]\n    lines = wikitext.splitlines()\n    start = None\n    for index, line in enumerate(lines):\n        if any(re.match(pattern, line.strip(), flags=re.I) for pattern in patterns):\n            start = index + 1\n            break\n    if start is None:\n        return ""\n\n    section_lines = []\n    for line in lines[start:]:\n        if re.match(r"^==[^=].*==\\s*$", line.strip()):\n            break\n        if line.strip().startswith(("{{", "|", "[[File:", "[[Image:")):\n            continue\n        section_lines.append(line)\n\n    plot = clean_wiki_text(" ".join(section_lines))\n    return plot[:2000].strip()\n\n\ndef clean_wiki_text(value: str) -> str:\n    value = re.sub(r"<!--.*?-->", "", value, flags=re.S)\n    value = re.sub(r"<ref[^>]*>.*?</ref>", "", value, flags=re.I | re.S)\n    value = re.sub(r"<ref[^/]*/>", "", value, flags=re.I)\n    value = re.sub(r"<br\\s*/?>", ", ", value, flags=re.I)\n    value = re.sub(r"<[^>]+>", "", value)\n    value = re.sub(r"\\{\\{(?:ubl|plainlist|plain list|unbulleted list)\\|", "", value, flags=re.I)\n    value = re.sub(r"\\{\\{nowrap\\|([^{}]+)\\}\\}", r"\\1", value)\n    value = re.sub(r"\\{\\{flag\\|([^{}|]+).*?\\}\\}", r"\\1", value)\n    value = re.sub(r"\\{\\{convert\\|([^{}|]+)\\|([^{}|]+)\\|?.*?\\}\\}", r"\\1 \\2", value, flags=re.I)\n    value = re.sub(r"\\{\\{(?:lang|native name|small|abbr)[^{}]*\\}\\}", "", value, flags=re.I)\n    value = re.sub(r"\\{\\{[^{}]+\\}\\}", "", value)\n    value = re.sub(r"\\{\\{[^{}]*", "", value)\n    value = re.sub(r"\\|group\\s*=\\s*[^}|]+", "", value)\n    value = re.sub(r"\\{\\{efn\\|.*", "", value)\n    value = re.sub(r"\\[\\[[^|\\]]+\\|([^\\]]+)\\]\\]", r"\\1", value)\n    value = re.sub(r"\\[\\[([^\\]]+)\\]\\]", r"\\1", value)\n    value = re.sub(r"\\[[a-z]+://[^\\s\\]]+\\s+([^\\]]+)\\]", r"\\1", value)\n    value = re.sub(r"\'{2,}", "", value)\n    value = value.replace("}}", "")\n    value = value.replace("&nbsp;", " ").replace("\\xa0", " ")\n    value = value.replace("*", ", ")\n    value = re.sub(r"\\s+", " ", value)\n    return value.strip(" ,|")\n\n\ndef build_row(candidate: dict, wiki_fields: dict) -> dict:\n    budget = parse_money_to_millions(wiki_fields.get("budget"))\n    box_office = parse_money_to_millions(wiki_fields.get("BoxOffice"))\n    return {\n        "tconst": candidate["tconst"],\n        "primaryTitle": candidate["primaryTitle"],\n        "startYear": candidate["startYear"],\n        "genres": candidate["genres"],\n        "lead_actors_ids": candidate.get("lead_actors_ids", "[]"),\n        "runtimeMinutes": candidate["runtimeMinutes"],\n        "averageRating": candidate["averageRating"],\n        "Language": wiki_fields.get("Language", ""),\n        "Country": wiki_fields.get("Country", ""),\n        "numVotes": candidate["numVotes"],\n        "budget": budget,\n        "BoxOffice": box_office,\n        "plot": wiki_fields.get("plot", ""),\n    }\n\n\ndef collect() -> list[dict]:\n    paths = ensure_imdb_files()\n    candidates = build_imdb_candidates(paths)\n\n    selected: list[tuple[dict, dict]] = []\n    for index, candidate in enumerate(candidates, start=1):\n        if len(selected) >= TARGET_MOVIE_COUNT:\n            break\n        if index % 100 == 0:\n            print(\n                f"Wikipedia enrichment progress: {len(selected)} collected from {index} candidates",\n                flush=True,\n            )\n\n        wiki_fields = {}\n        if index <= WIKIPEDIA_ENRICH_LIMIT:\n            page = wikipedia_search(candidate["primaryTitle"], candidate["startYear"])\n            if page:\n                wiki_fields = wikipedia_page_fields(page)\n\n        if REQUIRE_WIKIPEDIA_PAGE and not wiki_fields:\n            continue\n        selected.append((candidate, wiki_fields))\n\n    if len(selected) < TARGET_MOVIE_COUNT:\n        raise RuntimeError(\n            f"Collected only {len(selected)} movies. Increase CANDIDATE_POOL_SIZE "\n            f"above {CANDIDATE_POOL_SIZE} and run again."\n        )\n\n    selected_candidates = [candidate for candidate, _wiki_fields in selected]\n    attach_actor_ids(paths, selected_candidates)\n    rows = [build_row(candidate, wiki_fields) for candidate, wiki_fields in selected]\n    return rows\n\n\ndef write_dataset(rows: list[dict]) -> None:\n    with DATASET_PATH.open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef missing_stats(rows: list[dict]) -> dict:\n    stats = {}\n    total = len(rows)\n    for field in FIELDNAMES:\n        missing = 0\n        for row in rows:\n            value = row.get(field)\n            if value in (None, "", "[]") or (isinstance(value, float) and math.isnan(value)):\n                missing += 1\n        stats[field] = {\n            "missing_count": missing,\n            "missing_percent": round((missing / total * 100) if total else 0, 2),\n        }\n    return {\n        "movie_count": total,\n        "team_letter": TEAM_LETTER,\n        "target_movie_count": TARGET_MOVIE_COUNT,\n        "candidate_pool_size": CANDIDATE_POOL_SIZE,\n        "require_wikipedia_page": REQUIRE_WIKIPEDIA_PAGE,\n        "wikipedia_enrich_limit": WIKIPEDIA_ENRICH_LIMIT,\n        "wikipedia_enriched_count": sum(1 for row in rows if row.get("plot") or row.get("Language") or row.get("Country")),\n        "missing": stats,\n    }\n\n\ndef main() -> None:\n    CACHE_DIR.mkdir(exist_ok=True)\n    rows = collect()\n    write_dataset(rows)\n    stats = missing_stats(rows)\n    write_json(REPORT_STATS_PATH, stats)\n    print(f"Wrote {len(rows)} movies to {DATASET_PATH}", flush=True)\n    print(json.dumps(stats, indent=2, ensure_ascii=False), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('collect_movies.py').write_text(collect_movies_source, encoding='utf-8')


## Run Initial Collection

`collect_movies.main()` creates the initial IMDb base dataset and writes `dataset.csv` plus `report_stats.json`. The final submitted CSV is filtered to rows with Wikipedia enrichment. The collection settings keep only movies, years up to 2024, runtimes from 60 to 300 minutes, and primary titles beginning with `M`. When the submitted `dataset.csv` already exists, the next cell reuses it so the notebook can run quickly and reproducibly; if the CSV is missing, it runs the full collection.


In [ ]:
import collect_movies

dataset_path = Path('dataset.csv')
if dataset_path.exists():
    print('dataset.csv already exists; using the submitted final dataset.')
else:
    print('dataset.csv is missing; running the full IMDb/Wikipedia collection.')
    collect_movies.main()


## Resumable Wikipedia Enrichment

Wikipedia enrichment is intentionally separated into resumable chunks. It updates `dataset.csv` and `report_stats.json` after every checkpoint. Rows where Wikipedia has no usable data are recorded in `wiki_enrich_state.json` so later runs skip them instead of retrying the same rows forever.


In [ ]:
enrich_wikipedia_source = '#!/usr/bin/env python3\n"""Incrementally enrich dataset.csv with English Wikipedia fields.\n\nThis script is intentionally resumable. It only touches rows that still miss\nWikipedia fields, writes checkpoints, and updates report_stats.json after each\ncheckpoint. Stop it at any time and rerun the same command to continue.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport os\nimport urllib.parse\n\nimport collect_movies\n\n\nBATCH_SIZE = int(os.environ.get("ENRICH_BATCH_SIZE", "250"))\nSAVE_EVERY = int(os.environ.get("ENRICH_SAVE_EVERY", "10"))\nSEARCH_LIMIT = int(os.environ.get("WIKI_SEARCH_LIMIT", "1"))\nRETRY_FAILED = os.environ.get("ENRICH_RETRY_FAILED", "0").strip().lower() not in {"0", "false", "no"}\nSTATE_PATH = collect_movies.BASE_DIR / "wiki_enrich_state.json"\n\n\ndef has_wikipedia_fields(row: dict) -> bool:\n    return bool(row.get("plot") or row.get("Language") or row.get("Country"))\n\n\ndef row_key(row: dict) -> str:\n    return row.get("tconst") or f"{row.get(\'primaryTitle\', \'\')}:{row.get(\'startYear\', \'\')}"\n\n\ndef load_state() -> dict:\n    state = collect_movies.read_json(STATE_PATH)\n    failed = state.get("failed_rows")\n    if not isinstance(failed, dict):\n        failed = {}\n    return {"failed_rows": failed}\n\n\ndef write_state(state: dict) -> None:\n    collect_movies.write_json(STATE_PATH, state)\n\n\ndef combined_cache_path(row: dict) -> object:\n    query = f"{row[\'primaryTitle\']} {row[\'startYear\']} film"\n    return collect_movies.CACHE_DIR / f"wiki_combined_{SEARCH_LIMIT}_{collect_movies.safe_name(query)}.json"\n\n\ndef page_wikitext(page: dict) -> str:\n    revisions = page.get("revisions") or []\n    if not revisions:\n        return ""\n    slots = revisions[0].get("slots", {})\n    return slots.get("main", {}).get("content", "")\n\n\ndef wikipedia_combined_fields(row: dict) -> tuple[dict, str, bool]:\n    query = f"{row[\'primaryTitle\']} {row[\'startYear\']} film"\n    params = urllib.parse.urlencode({\n        "action": "query",\n        "generator": "search",\n        "gsrsearch": query,\n        "gsrlimit": SEARCH_LIMIT,\n        "prop": "revisions",\n        "rvprop": "content",\n        "rvslots": "main",\n        "format": "json",\n        "formatversion": 2,\n        "redirects": 1,\n        "utf8": 1,\n    })\n    cache_name = f"wiki_combined_{SEARCH_LIMIT}_{collect_movies.safe_name(query)}.json"\n    data = collect_movies.get_json(f"https://en.wikipedia.org/w/api.php?{params}", cache_name)\n    cache_exists = combined_cache_path(row).exists()\n    pages = data.get("query", {}).get("pages", [])\n    if not pages:\n        return {}, "no Wikipedia search result", cache_exists\n\n    empty_titles = []\n    for page in sorted(pages, key=lambda item: item.get("index", 999)):\n        page_title = page.get("title", "")\n        wikitext = page_wikitext(page)\n        if not wikitext:\n            empty_titles.append(page_title or "unknown page")\n            continue\n\n        fields = collect_movies.extract_infobox_fields(wikitext)\n        fields["plot"] = collect_movies.extract_plot(wikitext)\n        if fields.get("plot") or fields.get("Language") or fields.get("Country"):\n            return fields, page_title, True\n        empty_titles.append(page_title or "unknown page")\n\n    return {}, f"no usable Wikipedia fields in candidates: {\', \'.join(empty_titles[:5])}", cache_exists\n\n\ndef write_outputs(rows: list[dict]) -> None:\n    collect_movies.write_dataset(rows)\n    stats = collect_movies.missing_stats(rows)\n    collect_movies.write_json(collect_movies.REPORT_STATS_PATH, stats)\n\n\ndef enrich_row(row: dict) -> tuple[bool, str, bool]:\n    wiki_fields, page, should_skip_later = wikipedia_combined_fields(row)\n    if not wiki_fields:\n        return False, page, should_skip_later\n\n    row["Language"] = wiki_fields.get("Language", row.get("Language", ""))\n    row["Country"] = wiki_fields.get("Country", row.get("Country", ""))\n    row["budget"] = collect_movies.parse_money_to_millions(wiki_fields.get("budget"))\n    row["BoxOffice"] = collect_movies.parse_money_to_millions(wiki_fields.get("BoxOffice"))\n    row["plot"] = wiki_fields.get("plot", row.get("plot", ""))\n    changed = has_wikipedia_fields(row)\n    return changed, "enriched" if changed else f"empty usable fields on {page}", not changed\n\n\ndef main() -> int:\n    with collect_movies.DATASET_PATH.open(encoding="utf-8", newline="") as f:\n        rows = list(csv.DictReader(f))\n\n    state = load_state()\n    failed_rows = state["failed_rows"]\n    before = sum(1 for row in rows if has_wikipedia_fields(row))\n    attempted = 0\n    enriched = 0\n    skipped_failed = 0\n    newly_failed = 0\n    print(f"Starting Wikipedia enrichment: {before}/{len(rows)} rows already enriched", flush=True)\n\n    for index, row in enumerate(rows, start=1):\n        if attempted >= BATCH_SIZE:\n            break\n        if has_wikipedia_fields(row):\n            continue\n        key = row_key(row)\n        if key in failed_rows and not RETRY_FAILED:\n            skipped_failed += 1\n            continue\n        if RETRY_FAILED and failed_rows.get(key, {}).get("retry_search_limit", 0) >= SEARCH_LIMIT:\n            skipped_failed += 1\n            continue\n        if RETRY_FAILED and key not in failed_rows:\n            continue\n\n        attempted += 1\n        try:\n            changed, reason, should_skip_later = enrich_row(row)\n        except Exception as exc:\n            print(f"Row {index} {row[\'primaryTitle\']}: {type(exc).__name__}: {exc}", flush=True)\n            changed = False\n            reason = f"{type(exc).__name__}: {exc}"\n            should_skip_later = False\n\n        if changed:\n            enriched += 1\n            failed_rows.pop(key, None)\n        elif should_skip_later:\n            failed_rows[key] = {\n                "title": row.get("primaryTitle", ""),\n                "year": row.get("startYear", ""),\n                "reason": reason,\n                "retry_search_limit": SEARCH_LIMIT if RETRY_FAILED else 0,\n            }\n            newly_failed += 1\n\n        if attempted % SAVE_EVERY == 0:\n            write_outputs(rows)\n            write_state(state)\n            total = sum(1 for item in rows if has_wikipedia_fields(item))\n            print(\n                f"Checkpoint: attempted {attempted}/{BATCH_SIZE}, "\n                f"newly enriched {enriched}, newly failed {newly_failed}, "\n                f"skipped previous failures {skipped_failed}, "\n                f"total enriched {total}/{len(rows)}",\n                flush=True,\n            )\n\n    write_outputs(rows)\n    write_state(state)\n    total = sum(1 for row in rows if has_wikipedia_fields(row))\n    print(\n        f"Finished chunk: attempted {attempted}, newly enriched {enriched}, "\n        f"newly failed {newly_failed}, skipped previous failures {skipped_failed}, "\n        f"total enriched {total}/{len(rows)}",\n        flush=True,\n    )\n    return attempted\n\n\nif __name__ == "__main__":\n    main()\n'
Path('enrich_wikipedia.py').write_text(enrich_wikipedia_source, encoding='utf-8')


In [ ]:
# Run one chunk manually if needed. For overnight runs, use run_enrichment.sh from Terminal.
# import enrich_wikipedia
# enrich_wikipedia.main()


## Report Generation Code

The report script reads the current `dataset.csv`, recomputes missing-value statistics, writes `report_stats.json`, and creates the required short `report.pdf` with collection decisions, field descriptions, row counts, and missing-value percentages.


In [ ]:
make_report_source = '#!/usr/bin/env python3\n"""Create a short PDF report from the current dataset."""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport math\nfrom pathlib import Path\n\n\nBASE_DIR = Path(__file__).resolve().parent\nDATASET_PATH = BASE_DIR / "dataset.csv"\nSTATS_PATH = BASE_DIR / "report_stats.json"\nREPORT_PATH = BASE_DIR / "report.pdf"\nFIELDNAMES = [\n    "tconst",\n    "primaryTitle",\n    "startYear",\n    "genres",\n    "lead_actors_ids",\n    "runtimeMinutes",\n    "averageRating",\n    "Language",\n    "Country",\n    "numVotes",\n    "budget",\n    "BoxOffice",\n    "plot",\n]\n\n\ndef pdf_escape(text: str) -> str:\n    return text.replace("\\\\", "\\\\\\\\").replace("(", "\\\\(").replace(")", "\\\\)")\n\n\ndef wrap(text: str, width: int = 92) -> list[str]:\n    words = text.split()\n    lines = []\n    current = ""\n    for word in words:\n        candidate = f"{current} {word}".strip()\n        if len(candidate) > width and current:\n            lines.append(current)\n            current = word\n        else:\n            current = candidate\n    if current:\n        lines.append(current)\n    return lines\n\n\ndef missing_stats(rows: list[dict]) -> dict:\n    stats = {}\n    total = len(rows)\n    for field in FIELDNAMES:\n        missing = 0\n        for row in rows:\n            value = row.get(field)\n            if value in (None, "", "[]") or (isinstance(value, float) and math.isnan(value)):\n                missing += 1\n        stats[field] = {\n            "missing_count": missing,\n            "missing_percent": round((missing / total * 100) if total else 0, 2),\n        }\n\n    return {\n        "movie_count": total,\n        "team_letter": "M",\n        "require_wikipedia_page": False,\n        "wikipedia_enriched_count": sum(\n            1 for row in rows if row.get("plot") or row.get("Language") or row.get("Country")\n        ),\n        "wikipedia_core_complete_count": sum(\n            1 for row in rows if row.get("plot") and row.get("Language") and row.get("Country")\n        ),\n        "missing": stats,\n    }\n\n\ndef load_current_stats() -> dict:\n    with DATASET_PATH.open(encoding="utf-8", newline="") as f:\n        rows = list(csv.DictReader(f))\n    stats = missing_stats(rows)\n    STATS_PATH.write_text(json.dumps(stats, indent=2, ensure_ascii=False), encoding="utf-8")\n    return stats\n\n\ndef build_lines(stats: dict) -> list[str]:\n    lines = [\n        "Movie Dataset Collection - Part 1",\n        "",\n        "Collection decisions:",\n        f"Our assigned team letter is {stats.get(\'team_letter\', \'M\')}, so we collected only movies",\n        "whose primaryTitle starts with that letter. We used IMDb\'s non-commercial TSV",\n        "datasets, not the paid IMDb API. We kept titleType movie, release years up to",\n        "and including 2024, and runtimes between 60 and 300 minutes.",\n        "IMDb TSV files supplied the IMDb title id, title, year, genres, runtime, rating,",\n        "votes and actor ids. English Wikipedia pages were used for budget, worldwide",\n        "gross, language, country and plot when available. The final submitted CSV keeps",\n        "only rows with at least one Wikipedia-backed field and documents missing values",\n        "explicitly.",\n        "Rows without Wikipedia enrichment excluded from final CSV: True.",\n        f"Rows with at least one Wikipedia-backed field: {stats.get(\'wikipedia_enriched_count\', 0)}.",\n        f"Rows with Language, Country and plot all present: {stats.get(\'wikipedia_core_complete_count\', 0)}.",\n        "",\n        f"Number of rows in dataset.csv: {stats.get(\'movie_count\', 0)}",\n        "",\n        "Field description and source:",\n        "tconst: IMDb title id, from title.basics.tsv.gz.",\n        "primaryTitle: movie title, from title.basics.tsv.gz.",\n        "startYear: release year, from title.basics.tsv.gz.",\n        "genres: list of genres as supplied by title.basics.tsv.gz.",\n        "lead_actors_ids: up to five IMDb actor ids, from title.principals.tsv.gz.",\n        "runtimeMinutes: runtime in minutes, from title.basics.tsv.gz.",\n        "averageRating: IMDb average rating, from title.ratings.tsv.gz.",\n        "Language: primary language, from English Wikipedia.",\n        "Country: production country, from English Wikipedia.",\n        "numVotes: number of IMDb votes, from title.ratings.tsv.gz.",\n        "budget: production budget in USD millions, from Wikipedia when available.",\n        "BoxOffice: global box office in USD millions, from Wikipedia when available.",\n        "plot: plot summary, from English Wikipedia.",\n        "",\n        "Missing values by column:",\n    ]\n\n    for field, item in stats.get("missing", {}).items():\n        lines.append(f"{field}: {item[\'missing_count\']} missing ({item[\'missing_percent\']}%)")\n\n    lines += [\n        "",\n        "Notes:",\n        "Budget and worldwide gross are the least complete fields because many Wikipedia",\n        "infoboxes do not include them or write them in a format that is hard to normalize.",\n        "Some Wikipedia-derived values may contain extraction noise from inconsistent",\n        "infobox formatting, especially Language, Country, budget and BoxOffice.",\n        "All monetary values in dataset.csv are stored as USD millions.",\n    ]\n    return [line for item in lines for line in (wrap(item) if item else [""])]\n\n\ndef write_pdf(lines: list[str], path: Path) -> None:\n    # Minimal single-page PDF using built-in Helvetica. This avoids external packages.\n    content_lines = ["BT", "/F1 10 Tf", "50 790 Td", "14 TL"]\n    for line in lines[:54]:\n        content_lines.append(f"({pdf_escape(line)}) Tj")\n        content_lines.append("T*")\n    content_lines.append("ET")\n    stream = "\\n".join(content_lines).encode("latin-1", errors="replace")\n\n    objects = [\n        b"<< /Type /Catalog /Pages 2 0 R >>",\n        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",\n        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "\n        b"/Resources << /Font << /F1 4 0 R >> >> /Contents 5 0 R >>",\n        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",\n        b"<< /Length " + str(len(stream)).encode("ascii") + b" >>\\nstream\\n" + stream + b"\\nendstream",\n    ]\n\n    pdf = bytearray(b"%PDF-1.4\\n")\n    offsets = [0]\n    for index, obj in enumerate(objects, start=1):\n        offsets.append(len(pdf))\n        pdf.extend(f"{index} 0 obj\\n".encode("ascii"))\n        pdf.extend(obj)\n        pdf.extend(b"\\nendobj\\n")\n    xref = len(pdf)\n    pdf.extend(f"xref\\n0 {len(objects) + 1}\\n".encode("ascii"))\n    pdf.extend(b"0000000000 65535 f \\n")\n    for offset in offsets[1:]:\n        pdf.extend(f"{offset:010d} 00000 n \\n".encode("ascii"))\n    pdf.extend(\n        f"trailer << /Size {len(objects) + 1} /Root 1 0 R >>\\nstartxref\\n{xref}\\n%%EOF\\n".encode("ascii")\n    )\n    path.write_bytes(pdf)\n\n\ndef main() -> None:\n    stats = load_current_stats()\n    write_pdf(build_lines(stats), REPORT_PATH)\n    print(f"Wrote {REPORT_PATH}")\n\n\nif __name__ == "__main__":\n    main()\n'
Path('make_report.py').write_text(make_report_source, encoding='utf-8')


In [ ]:
import make_report

make_report.main()


## Dataset Validation

These checks verify the assignment constraints: required columns, more than 5,000 final CSV rows, only `M` titles, years through 2024, runtimes from 60 to 300 minutes, and Wikipedia enrichment for every included row.


In [ ]:
import csv, json

required_columns = ['tconst', 'primaryTitle', 'startYear', 'genres', 'lead_actors_ids', 'runtimeMinutes', 'averageRating', 'Language', 'Country', 'numVotes', 'budget', 'BoxOffice', 'plot']
with Path('dataset.csv').open(encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))

non_m_titles = sum(1 for row in rows if not row['primaryTitle'].upper().startswith('M'))
max_year = max(int(row['startYear']) for row in rows)
runtime_min = min(int(row['runtimeMinutes']) for row in rows)
runtime_max = max(int(row['runtimeMinutes']) for row in rows)
wikipedia_enriched = sum(1 for row in rows if row.get('plot') or row.get('Language') or row.get('Country'))

assert len(rows) >= 5000
assert list(rows[0].keys()) == required_columns
assert non_m_titles == 0
assert max_year <= 2024
assert runtime_min >= 60 and runtime_max <= 300
assert wikipedia_enriched == len(rows)

print(f"Rows: {len(rows)}")
print('Columns correct:', list(rows[0].keys()) == required_columns)
print('Non-M titles:', non_m_titles)
print('Max year:', max_year)
print('Runtime range:', runtime_min, '-', runtime_max)
print('Wikipedia-enriched rows:', wikipedia_enriched)


In [ ]:
stats = json.loads(Path('report_stats.json').read_text(encoding='utf-8'))
print('Movie count:', stats['movie_count'])
for column, item in stats['missing'].items():
    print(f"{column}: {item['missing_count']} missing ({item['missing_percent']}%)")
